# 11. Анализ пригодности модели регрессии

### Нелинейная регрессия: зависимость энергии частицы от номера канала

Имеются экспериментальные данные о зависимости номера канала, в котором регистрируется частица, от энергии частицы:

| E, кэВ | 75,99 | 91,97 | 105,71 | 123,20 | 131,67 | 150,70 | 179,32 | 203,21 |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| p, канал | 1900 | 1910 | 1920 | 1930 | 1940 | 1950 | 1960 | 1970 |

1. Предложите вид нелинейной зависимости.
2. Оцените параметры зависимости.
3. Предложили ли Вы адекватную экспериментальным данным зависимость? Почему?
4. Ответ сопроводите графиками исследуемых зависимостей.

Дополнительные вопросы:
1. Проверить пригодность модели по критерию хи-квадрат (если такое возможно).
2. Рассчитать коэффициент множественной корреляции.
3. Рассчитать коэффициент детерминации.
4. Пригодна ли модель, которая использовалась для фитирования?

---
# Решение

Импортируем необходимые библиотеки и зададим исходные данные.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

channel = np.array([1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970])
energy  = np.array([75.99, 91.97, 105.71, 123.20, 131.67, 150.70, 179.32, 203.21])

n = len(channel)   # число наблюдений
p = 3              # число параметров модели (a, b, c)

print("Экспериментальные данные:")
print(f"  номер канала: {channel}")
print(f"  энергия, кэВ: {energy}")

## 1. Выбор вида зависимости и оценка параметров

Анализ экспериментальных данных показывает, что при равномерном увеличении номера канала на 10 единиц прирост регистрируемой энергии монотонно возрастает. Это свидетельствует о нелинейном (ускоряющемся) характере зависимости $E(p)$, что исключает применение линейной модели. В качестве аппроксимирующей функции предлагается полином второй степени:

$$E = a \cdot p^2 + b \cdot p + c$$

где $p$ — номер канала, $a$, $b$, $c$ — неизвестные параметры.

Для оценки параметров применяется метод наименьших квадратов (МНК). Запишем систему наблюдений в матричном виде $\mathbf{y} = X\boldsymbol{\theta} + \boldsymbol{\varepsilon}$, где $\boldsymbol{\varepsilon}$ — вектор случайных ошибок, а матрица плана:

$$X = \begin{pmatrix} p_1^2 & p_1 & 1 \\ \vdots & \vdots & \vdots \\ p_n^2 & p_n & 1 \end{pmatrix}$$

МНК-оценка вектора параметров $\hat{\boldsymbol{\theta}} = (\hat{a}, \hat{b}, \hat{c})^\top$, минимизирующая сумму квадратов отклонений $\sum_{i=1}^n \varepsilon_i^2$, определяется из нормальных уравнений:

$$\hat{\boldsymbol{\theta}} = (X^\top X)^{-1} X^\top \mathbf{y}$$

In [ ]:
X = np.column_stack((channel**2, channel, np.ones(n)))

theta = np.linalg.inv(X.T @ X) @ (X.T @ energy)
a, b, c = theta

print("Оцененные параметры:")
print(f"  a = {a:.6f} кэВ/кан^2")
print(f"  b = {b:.4f} кэВ/кан")
print(f"  c = {c:.2f} кэВ")

## 2. Остатки и стандартная ошибка регрессии

После нахождения оценок $\hat{a}$, $\hat{b}$, $\hat{c}$ вычислим предсказанные значения $\hat{E}_i = \hat{a} p_i^2 + \hat{b} p_i + \hat{c}$ и остатки регрессии:

$$r_i = E_i - \hat{E}_i, \quad i = 1, \ldots, n$$

Остатки характеризуют отклонение модели от наблюдений. Несмещённая оценка дисперсии случайной ошибки:

$$S_y^2 = \frac{\sum_{i=1}^n r_i^2}{n - p}$$

где $n = 8$ — число наблюдений, $p = 3$ — число оцениваемых параметров; знаменатель $n - p = 5$ — число степеней свободы остатков.

In [ ]:
energy_pred = a * channel**2 + b * channel + c
residuals   = energy - energy_pred

S2_y = np.sum(residuals**2) / (n - p)
S_y  = np.sqrt(S2_y)

print(f"Стандартная ошибка регрессии: S_y = {S_y:.4f} кэВ")
print(f"Максимальное абсолютное отклонение: {np.max(np.abs(residuals)):.4f} кэВ")

## 3. Проверка пригодности модели

### 3.1. Критерий хи-квадрат

Классический критерий хи-квадрат для проверки адекватности модели имеет вид:

$$\chi^2 = \sum_{i=1}^n \frac{(E_i - \hat{E}_i)^2}{\sigma_i^2}$$

где $\sigma_i^2$ — известная дисперсия $i$-го измерения. **Применение данного критерия в данной задаче невозможно**, поскольку индивидуальные дисперсии $\sigma_i^2$ не заданы и не могут быть оценены при единственном измерении в каждой точке. При $n - p = 5$ степенях свободы и отсутствии повторных наблюдений критерий хи-квадрат неприменим.

### 3.2. Коэффициент множественной корреляции и коэффициент детерминации

**Коэффициент детерминации** $R^2$ показывает долю дисперсии отклика, объяснённую моделью:

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}, \quad SS_{\text{res}} = \sum_{i=1}^n r_i^2, \quad SS_{\text{tot}} = \sum_{i=1}^n (E_i - \bar{E})^2$$

Значение $R^2 \in [0, 1]$; модель считается приемлемой при $R^2 \geq 0{,}95$.

**Скорректированный коэффициент детерминации** вводит штраф за увеличение числа параметров, что позволяет сравнивать модели с разным числом предикторов:

$$R^2_{\text{adj}} = 1 - (1 - R^2)\frac{n - 1}{n - p}$$

**Коэффициент множественной корреляции** $R$ — корень из $R^2$, численно равный коэффициенту корреляции Пирсона между наблюдаемыми и предсказанными значениями:

$$R = \mathrm{corr}(E,\, \hat{E})$$

In [ ]:
R_multiple = np.corrcoef(energy, energy_pred)[0, 1]
SS_res     = np.sum(residuals**2)
SS_tot     = np.sum((energy - np.mean(energy))**2)
R2         = 1 - SS_res / SS_tot
R2_adj     = 1 - (1 - R2) * (n - 1) / (n - p)

print(f"Коэффициент множественной корреляции: R     = {R_multiple:.6f}")
print(f"Коэффициент детерминации:             R^2   = {R2:.6f}")
print(f"Скорректированный коэффициент:        R^2adj = {R2_adj:.6f}")
print(f"Модель объясняет {R2 * 100:.2f}% вариации энергии")

## 4. Анализ остатков

Анализ остатков позволяет выявить систематические отклонения, нарушение гомоскедастичности и выбросы. Для корректно специфицированной модели остатки должны удовлетворять следующим условиям:

- среднее остатков $\bar{r} \approx 0$ — отсутствие систематической ошибки;
- отсутствие выраженных трендов или паттернов при построении $r_i$ против $p_i$ — подтверждение правильного выбора формы модели;
- $\max|r_i| \leq 3 S_y$ — отсутствие значимых выбросов.

Знаки остатков должны распределяться случайно; их систематическая группировка или чередование указывает на неверно выбранный вид аппроксимирующей функции.

In [ ]:
print(f"Среднее остатков: {np.mean(residuals):.6f} кэВ")
print()
print("Остатки по точкам:")
for i, (ch, res) in enumerate(zip(channel, residuals)):
    print(f"  точка {i+1}: канал {ch}, остаток = {res:.4f} кэВ")
print()

n_plus  = np.sum(residuals > 0)
n_minus = np.sum(residuals < 0)
print(f"Положительных остатков: {n_plus}, отрицательных: {n_minus}")
if abs(n_plus - n_minus) <= 2:
    print("Распределение знаков остатков примерно симметрично")
else:
    print("Наблюдается заметное неравенство знаков остатков")

## 5. Вывод о пригодности модели

Модель признаётся пригодной при одновременном выполнении трёх условий:

- $R^2 \geq 0{,}95$ — модель объясняет не менее 95% дисперсии отклика;
- $|\bar{r}| \ll S_y$ — систематическая ошибка отсутствует;
- $\max|r_i| \leq 3 S_y$ — нет статистически значимых выбросов.

In [ ]:
is_good = True
reasons = []

if R2 < 0.95:
    is_good = False
    reasons.append(f"R^2 = {R2:.4f} < 0.95")
if np.abs(np.mean(residuals)) > S_y / 2:
    is_good = False
    reasons.append(f"среднее остатков ({np.mean(residuals):.4f}) не близко к нулю")
if np.max(np.abs(residuals)) > 3 * S_y:
    is_good = False
    reasons.append(f"выброс: max|r| = {np.max(np.abs(residuals)):.3f} > 3*S_y")

if is_good:
    print("Модель ПРИГОДНА для описания экспериментальных данных")
    print(f"  R^2 = {R2:.6f} -- модель объясняет {R2*100:.2f}% дисперсии данных")
    print(f"  R   = {R_multiple:.6f} -- очень тесная связь")
    print(f"  S_y = {S_y:.4f} кэВ -- стандартная ошибка мала")
    print(f"  Максимальное отклонение: {np.max(np.abs(residuals)):.4f} кэВ")
else:
    print("Модель НЕ пригодна:")
    for r in reasons:
        print(f"  - {r}")

## 6. Графики

Для визуальной оценки качества подгонки строятся два графика:
- **левый**: экспериментальные точки и кривая квадратичной регрессии в координатах $(p,\, E)$;
- **правый**: диаграмма рассеяния остатков $r_i$ против номера канала $p_i$ с нулевой линией. Равномерное расположение точек вокруг нуля без выраженного тренда подтверждает адекватность выбранной модели.

In [ ]:
k_plot = np.linspace(1890, 1980, 200)
E_plot = a * k_plot**2 + b * k_plot + c

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(channel, energy, color="blue", s=80, label="эксперимент")
axes[0].plot(k_plot, E_plot, "r-", lw=2, label="квадратичная регрессия")
axes[0].set_xlabel("номер канала")
axes[0].set_ylabel("энергия частицы, кэВ")
axes[0].set_title("Зависимость энергии от номера канала")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(channel, residuals, color="red", s=60)
axes[1].axhline(y=0, color="black", linestyle="--")
axes[1].set_xlabel("номер канала")
axes[1].set_ylabel("остатки, кэВ")
axes[1].set_title("График остатков квадратичной модели")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()